## testing model

In [ ]:
import sentence_transformers as st
import numpy as np
import pandas as pd
import os, sys

sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from confluence_api.getting_confluence_content import *

In [ ]:
documentation = pd.read_json(os.path.join(os.path.dirname(os.getcwd()), 'data', 'documentation_embeddings.json'))
# model = st.SentenceTransformer('model_2')
model = st.SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1')

In [ ]:
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
def search_documents(searched_sentence, documentation, model):
    df = documentation[['note']]

    searched_sent_enc = model.encode(searched_sentence)
    # df['similarity_score'] = documentation.embeddings.apply(lambda x: np.max([cos_sim(x[i], searched_sent_enc) for i in range(len(x))]))
    df['similarity_score'] = documentation.embeddings.apply(lambda x: np.max([np.dot(x[i], searched_sent_enc) for i in range(len(x))]))
    return df.sort_values(by = 'similarity_score', ascending = False).iloc[ : 10].reset_index(drop = True)

In [ ]:
def divide_text(text, chunk_size):
    # chunk_size is a number of words per chunk
    chunks = []
    for i in range(
        0, 
        len(text.split()) - (len(text.split()) % chunk_size), 
        chunk_size // 2
    ):
        chunks.append(' '.join(text.split()[i : i + chunk_size]))

    chunks.append(' '.join(text.split()[len(text.split()) - chunk_size : ]))
    
    return chunks

In [ ]:
# questions with which original model has problems


# searched_sentence = 'translators worked hours'
# searched_sentence = 'how much translators worked'
# searched_sentence = 'data about translators work efficiency'

# searched_sentence = 'data about vendors'
# searched_sentence = 'data about suppliers'
# searched_sentence = 'data about providers'

# searched_sentence = 'helix data procurements'

# searched_sentence = 'cant see productivity data'
searched_sentence = 'productive hours worked are not visible in insight'

notes = search_documents(searched_sentence, documentation, model)
for i, (note, score) in notes.iterrows():
    print(f'{i}) similarity score:', score)
    print(note, '\n')

In [ ]:
# searched_sentence = 'translators worked hours'
# searched_sentence = 'how much translators worked'
# searched_sentence = 'data about translators work efficiency'

# searched_sentence = 'where I can find data about productivity metric'
# searched_sentence = 'where I can find data about efficiency metric'

# searched_sentence = 'data about vendors'
# searched_sentence = 'data about suppliers'
# searched_sentence = 'data about providers'

# searched_sentence = 'information about professions'

# searched_sentence = 'how many employees we fired'
# searched_sentence = 'how many workers we hired'

# searched_sentence = 'helix data procurements'
# searched_sentence = 'helix data purchases'
# searched_sentence = 'helix data orders'

# searched_sentence = 'data about helix tasks'
# searched_sentence = 'data about helix jobs'

# searched_sentence = 'who can help with helix data'

# searched_sentence = 'help with sisense problems'

# searched_sentence = "can't open dashboards"

# searched_sentence = 'where I can learn about our company'

# searched_sentence = 'how to update headcount dashboard'

# searched_sentence = 'information about how good are our translations'
# searched_sentence = 'information about how good is our work'
# searched_sentence = 'information about how well we do our work'

# searched_sentence = 'cant see productivity data'
# searched_sentence = 'productive hours worked are not visible in insight'
# searched_sentence = 'make hours worked visible in insight'

# searched_sentence = 'introduction sisense'
# searched_sentence = 'learning sisense'
# searched_sentence = 'sisense tutorial'

# searched_sentence = 'passwords systems'
# searched_sentence = 'access systems'
# searched_sentence = 'passwords websites'
# searched_sentence = 'systems cridentials'

# searched_sentence = 'order bar plot'

# searched_sentence = 'sisense graphs widgets'

# searched_sentence = 'order columns pivot table'

searched_sentence = 'logic behind grand total'

notes = search_documents(searched_sentence, documentation, model)
for i, (note, score) in notes.iterrows():
    print(f'{i}) similarity score:', score)
    print(note, '\n')

0) similarity score: 19.047389124016988
https://confluence.sdl.com/display/DA/Sort+Column+chart+with+Break-by+values 

1) similarity score: 17.148988694675676
table dw.fact.Productivity_TIME contains information about how much time employees were working 

2) similarity score: 14.918322250091425
confluence.sdl.com/display/DA/Blox+widget+for+modifying+a+pivot+table is a link with a video showing how to use and create blox widget which allows you to choose a data for rows and values in pivot table.
If you don't have access to widgets from folder 'Blox Repository' showed in the video then write to me. 

3) similarity score: 14.892914183930587
table Stage.dbi.AdjustmentFactor_Map contains information about adjustment factors for calculating adjusted words for different tasks and units 

4) similarity score: 14.39113963680459
data about how much productive hours or words someone has is taken from the table stage.emp.tbl_TSProductivityTimeSpans and stage.emp.tbl_TSProductivityEntries 

5) si

In [ ]:
# checking similarity between searched_sentence and text content from confluence page page_title 
searched_sentence = 'sisense onboarding'
page_title = 'Sisense On-boarding Information'

searched_sent_enc = model.encode(searched_sentence)

page_id = get_page_info(page_title = page_title)['id']
text = get_text_content(page_id)

# checking similarity between searched sentence and different chunks of confluence page text content
chunks = divide_text(text, 5)
sim_scores = []
for chunk in chunks:
    chunk_enc = model.encode(chunk)
    sim_scores.append([cos_sim(chunk_enc, searched_sent_enc), chunk])
    
sim_scores = pd.DataFrame(sim_scores).sort_values(by = 0, ascending = False).reset_index(drop = True)
sim_scores.columns = ['similarity_score', 'sentence']

0.3313057

In [ ]:
# checking similarity between searched_sentence and different chunks of text
searched_sentence = 'how to control grand total'
text = """
The logic behind Grand Total is indeed complex as it uses the same formula as in the "Values". So if you add a grand total for the column that uses SUM() function, Grand total itself will also do a SUM() for all cells. Essentially grand total applies the same calculation/formulas but without aggregation/group by.

There is a way to control the logic behind Grand Total though. You can click on the 3-dot menu next to the "Value" and change the value for the "Subtotal By" from Default (Mimic formula) to a specific function like SUM, MIN, MAX etc.
See more details on this here:
https://docs.sisense.com/main/SisenseLinux/pivot.htm?Highlight=grand%20total#GrandTotalsandSubtotals

Note, the "Default" logic of totals is applied without LIMIT, meaning if you have 10,000 rows in Pivot and 20,000 rows in the datasource the calculation will be based on all 20,000 rows. But if you switch from Default to anything else, say Average, the total calculation will be done for 10,000 rows visible in Pivot. 
"""

searched_sent_enc = model.encode(searched_sentence)
# checking similarity between searched sentence and different chunks of confluence page text content
chunks = divide_text(text, 6)
sim_scores = []
for chunk in chunks:
    chunk_enc = model.encode(chunk)
    sim_scores.append([cos_sim(chunk_enc, searched_sent_enc), chunk])
    
sim_scores = pd.DataFrame(sim_scores).sort_values(by = 0, ascending = False).reset_index(drop = True)
sim_scores.columns = ['similarity_score', 'sentence']

In [ ]:
for i, (score, sentence) in sim_scores.iloc[:5].iterrows():
    print(f'{i}) score:', score)
    print(sentence, '\n')

0) score: 0.7183789014816284
add a grand total for the 

1) score: 0.6631654500961304
The logic behind Grand Total is 

2) score: 0.6401830315589905
the logic behind Grand Total though. 

3) score: 0.6315935254096985
SUM() function, Grand total itself will 

4) score: 0.602814257144928
There is a way to control 



In [ ]:
sent_enc = model.encode("mom")
searched_sent_enc = model.encode("dad")
cos_sim(sent_enc, searched_sent_enc)

0.8219362